In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime
import pandas as pd
from sigen_api import sigen_authenticate, get_dfRebanhoExame
from ipydatagrid import DataGrid
import io
from IPython.display import HTML
import openpyxl
import base64

# Sessão global
sessao_atual = {}

# --- HEADER ---
def carregar_logo(url):
    import requests
    img = widgets.Image(layout=widgets.Layout(width='300px', height='auto'))
    try:
        r = requests.get(url)
        if r.status_code == 200:
            img.value = r.content
    except:
        pass
    return img

logo_url = "https://www.cidasc.sc.gov.br/wp-content/themes/cidasc/images/logo_topo_nomal_2023.png"
logo_img = carregar_logo(logo_url)
titulo = widgets.HTML("<h1 style='margin:0;'>BTCerti</h1>")
subtitulo = widgets.HTML("<h3 style='margin:0;'>Análise de Processos de Certificação do PNCEBT - CIDASC</h3>")

header = widgets.VBox(
    [logo_img, titulo, subtitulo],
    layout=widgets.Layout(align_items='center', width='100%', padding='10px')
)

# --- TELA DE LOGIN ---
label_usuario = widgets.HTML("<b>Usuário:</b>")
campo_usuario = widgets.Text(
    placeholder='Digite aqui seu usuário do Sigen+',
    layout=widgets.Layout(width='300px')
)

label_senha = widgets.HTML("<b>Senha:</b>")
campo_senha = widgets.Password(
    placeholder='Digite aqui sua senha do Sigen+',
    layout=widgets.Layout(width='300px')
)

botao_entrar = widgets.Button(
    description='Entrar',
    icon='sign-in-alt',
    layout=widgets.Layout(width='150px')
)

msg_erro_login = widgets.HTML("", layout=widgets.Layout(visibility='hidden'))

form_login = widgets.VBox(
    [label_usuario, campo_usuario, label_senha, campo_senha, botao_entrar, msg_erro_login],
    layout=widgets.Layout(align_items='center')
)

# --- TELA PRINCIPAL ---
label_codigo = widgets.HTML("<b>Código Oficial:</b>")
campo_codigo = widgets.IntText(layout=widgets.Layout(width='300px'))

label_data = widgets.HTML("<b>Data (formato: DD/MM/AAAA):</b>")
campo_data = widgets.Text(
    placeholder='DD/MM/AAAA',
    layout=widgets.Layout(width='300px')
)

botao_relatorio = widgets.Button(description='Gerar Relatório', button_style='primary', icon='search')
botao_sair = widgets.Button(description='Sair', icon='sign-out-alt', button_style='danger')

saida_resultado = widgets.Output()

area_entrada_dados = widgets.VBox(
    [label_codigo, campo_codigo, label_data, campo_data, botao_relatorio, botao_sair],
    layout=widgets.Layout(align_items='center')
)

# Função para habilitar/desabilitar tela principal
def habilitar_tela_principal(habilitar):
    campo_codigo.disabled = not habilitar
    campo_data.disabled = not habilitar
    botao_relatorio.disabled = not habilitar
    botao_sair.disabled = not habilitar
    if not habilitar:
        saida_resultado.clear_output()

# Inicialmente tela principal desabilitada
habilitar_tela_principal(False)

# --- CALLBACKS ---
def ao_clicar_entrar(b):
    usuario = campo_usuario.value
    senha = campo_senha.value
    auth = sigen_authenticate(usuario, senha)

    if auth.get('login_error'):
        msg_erro_login.value = f"<span style='color:red;'>Erro de login: {auth.get('error_message')}</span>"
        msg_erro_login.layout.visibility = 'visible'
    else:
        sessao_atual['session'] = auth['session']
        msg_erro_login.value = ""
        msg_erro_login.layout.visibility = 'hidden'

        campo_usuario.disabled = True
        campo_senha.disabled = True
        botao_entrar.disabled = True
        botao_entrar.description = "Logado"
        botao_entrar.icon = "check"

        habilitar_tela_principal(True)

def ao_clicar_sair(b):
    sessao_atual.clear()
    campo_usuario.value = ""
    campo_senha.value = ""
    msg_erro_login.value = ""
    msg_erro_login.layout.visibility = 'hidden'

    campo_usuario.disabled = False
    campo_senha.disabled = False
    botao_entrar.disabled = False
    botao_entrar.description = "Entrar"
    botao_entrar.icon = "sign-in-alt"

    habilitar_tela_principal(False)

def normalizar_data(data_str):
    data_str = data_str.strip()

    if '/' in data_str:
        try:
            dt = datetime.strptime(data_str, '%d/%m/%Y')
            return dt.strftime('%d/%m/%Y')
        except ValueError:
            return None
    else:
        if len(data_str) == 8 and data_str.isdigit():
            try:
                dt = datetime.strptime(data_str, '%d%m%Y')
                return dt.strftime('%d/%m/%Y')
            except ValueError:
                return None
        else:
            return None

def ao_clicar_gerar(b):
    saida_resultado.clear_output()
    with saida_resultado:
        data_texto = campo_data.value.strip()
        cd = campo_codigo.value

        if not data_texto:
            print("Data não informada.")
            return

        data_formatada = normalizar_data(data_texto)
        if not data_formatada:
            print("Formato de data inválido. Use DDMMYYYY ou DD/MM/YYYY.")
            return

        if not cd:
            print("Código Oficial não informado.")
            return

        print(f"Data selecionada: {data_formatada}")
        print(f"Código oficial: {cd}")
        print('Gerando relatório, aguarde...')

        session = sessao_atual.get('session')
        resultado = get_dfRebanhoExame(cd, data_formatada, session)

        if resultado['dfRebanhoExame'].empty:
            print("Nenhum dado retornado.")
        else:
            saida_resultado.clear_output()
            print(f"Data selecionada: {data_formatada}")
            print(f"Código oficial: {cd}")

            df = resultado['dfRebanhoExame'].fillna("--")
            grid = DataGrid(df,
                             selection_mode='row',
                               layout={'height': '400px'},
                                 auto_fit_columns=True)
            
            grid.show_key = False
            grid.frozen_columns = 3
            display(grid)
            print('Relatório Gerado com sucesso!')

            # Geração de arquivo com nome único
            dt_obj = datetime.strptime(data_formatada, "%d/%m/%Y")
            dt_str = dt_obj.strftime('%d%m%Y')
            nome_arquivo = f"rel_{cd}-{dt_str}.xlsx"

            df_bytes = io.BytesIO()
            df.to_excel(df_bytes, index=False)
            df_bytes.seek(0)

            b64 = base64.b64encode(df_bytes.read()).decode()

            href = f'''
            <a download="{nome_arquivo}" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64}" target="_blank" style="font-weight:bold; font-size:16px;">
            📥 Baixar Excel
            </a>
            '''

            display(HTML(href))

# Liga callbacks
botao_entrar.on_click(ao_clicar_entrar)
botao_sair.on_click(ao_clicar_sair)
botao_relatorio.on_click(ao_clicar_gerar)

# Exibe tudo logo de cara (login habilitado, tela principal desabilitada)
display(header, form_login, area_entrada_dados, saida_resultado)


Output()